In [1]:
import pandas as pd
import numpy as np
import re

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Load dataset
df = pd.read_csv("../data/data_gojek.csv")
df.head()


,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion
0,e87c53c2-9d1e-4c70-8c51-178a09d17723,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"Untuk layanan goride, bila pengantarannya sepe...",5,524,5.27.2,2025-08-14 20:45:24,NaN,NaN,5.27.2
1,738f14e1-6d99-4bb6-a803-6d3d0f795aee,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,ongkir food mahal.. masa jarak dekat 17 K gak ...,1,37,5.29.2,2025-09-01 12:53:32,"Mohon maaf atas ketidaknyamanannya, Kak Imam. ...",2025-08-12 14:57:37,5.29.2
2,4f6d3efd-acbb-4204-8638-7a028132a244,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,Saya pernah gak sengaja kepencet mode hemat oj...,5,5,5.29.2,2025-09-02 08:21:40,NaN,NaN,5.29.2
3,494c634d-960a-4dc7-8f43-12d36625863b,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"Ada pengumpulan koin sekarang tiap hari, tapi ...",2,45,5.27.2,2025-08-13 06:50:40,"Hai Kak, mohon maaf atas kendalanya. Pastikan ...",2023-10-06 19:08:40,5.27.2
4,5bb845de-6f3e-448e-a2cd-47d09ecc9d33,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,klu sdh sering berlangganan jenis opsi apapun ...,3,62,5.27.2,2025-08-15 06:20:31,"Mohon maaf atas ketidaknyamanannya, Kak Ani. T...",2025-08-15 10:53:21,5.27.2


In [2]:
def preprocess_text(text):
  if pd.isna(text):
    return ""
  text = str(text).lower()
  text = re.sub(r'[^a-zA-Z\s]', '', text)
  text = re.sub(r'\s+', ' ', text).strip()
  return text

df['content'] = df['content'].apply(preprocess_text)

# Mapping kolom score ke sentiment
def map_score_to_sentiment(score):
    if pd.isna(score):
        return "NEUTRAL"
    if score >= 4:
        return "POSITIVE"
    elif score <= 2:
        return "NEGATIVE"
    else:
        return "NEUTRAL"

df["label"] = df["score"].apply(map_score_to_sentiment)
df.head()

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion,label
0,e87c53c2-9d1e-4c70-8c51-178a09d17723,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,untuk layanan goride bila pengantarannya seper...,5,524,5.27.2,2025-08-14 20:45:24,NaN,NaN,5.27.2,POSITIVE
1,738f14e1-6d99-4bb6-a803-6d3d0f795aee,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,ongkir food mahal masa jarak dekat k gak ada m...,1,37,5.29.2,2025-09-01 12:53:32,"Mohon maaf atas ketidaknyamanannya, Kak Imam. ...",2025-08-12 14:57:37,5.29.2,NEGATIVE
2,4f6d3efd-acbb-4204-8638-7a028132a244,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,saya pernah gak sengaja kepencet mode hemat oj...,5,5,5.29.2,2025-09-02 08:21:40,NaN,NaN,5.29.2,POSITIVE
3,494c634d-960a-4dc7-8f43-12d36625863b,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,ada pengumpulan koin sekarang tiap hari tapi h...,2,45,5.27.2,2025-08-13 06:50:40,"Hai Kak, mohon maaf atas kendalanya. Pastikan ...",2023-10-06 19:08:40,5.27.2,NEGATIVE
4,5bb845de-6f3e-448e-a2cd-47d09ecc9d33,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,klu sdh sering berlangganan jenis opsi apapun ...,3,62,5.27.2,2025-08-15 06:20:31,"Mohon maaf atas ketidaknyamanannya, Kak Ani. T...",2025-08-15 10:53:21,5.27.2,NEUTRAL


In [3]:

X = df["content"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)


In [4]:
models = {
    "SVM": Pipeline([
        ("tfidf", TfidfVectorizer(max_features=5000, stop_words="english")),
        ("clf", SVC(kernel="linear", probability=True, random_state=42))
    ]),
    "Naive Bayes": Pipeline([
        ("tfidf", TfidfVectorizer(max_features=5000, stop_words="english")),
        ("clf", MultinomialNB())
    ]),
    "Random Forest": Pipeline([
        ("tfidf", TfidfVectorizer(max_features=5000, stop_words="english")),
        ("clf", RandomForestClassifier(n_estimators=100, random_state=42))
    ])
}


In [5]:
results = {}

for name, model in models.items():
    print(f"\n=== {name} ===")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    print("Accuracy:", acc)
    print("Classification Report:\n", classification_report(y_test, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

    results[name] = {
        "model": model,
        "accuracy": acc,
        "report": classification_report(y_test, y_pred, output_dict=True)
    }



=== SVM ===
Accuracy: 0.7676666666666667
Classification Report:
               precision    recall  f1-score   support

    NEGATIVE       0.77      0.95      0.85      1902
     NEUTRAL       0.50      0.00      0.01       329
    POSITIVE       0.77      0.64      0.70       769

    accuracy                           0.77      3000
   macro avg       0.68      0.53      0.52      3000
weighted avg       0.74      0.77      0.72      3000

Confusion Matrix:
 [[1808    1   93]
 [ 277    1   51]
 [ 275    0  494]]

=== Naive Bayes ===
Accuracy: 0.737
Classification Report:
               precision    recall  f1-score   support

    NEGATIVE       0.71      0.99      0.83      1902
     NEUTRAL       0.00      0.00      0.00       329
    POSITIVE       0.94      0.41      0.58       769

    accuracy                           0.74      3000
   macro avg       0.55      0.47      0.47      3000
weighted avg       0.69      0.74      0.67      3000

Confusion Matrix:
 [[1892    0   10]


c:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\ProgramData\anaconda3\Lib\site-packages\sklearn\metr

Accuracy: 0.746
Classification Report:
               precision    recall  f1-score   support

    NEGATIVE       0.73      0.98      0.83      1902
     NEUTRAL       0.00      0.00      0.00       329
    POSITIVE       0.86      0.49      0.63       769

    accuracy                           0.75      3000
   macro avg       0.53      0.49      0.49      3000
weighted avg       0.68      0.75      0.69      3000

Confusion Matrix:
 [[1861    0   41]
 [ 311    0   18]
 [ 392    0  377]]


c:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\ProgramData\anaconda3\Lib\site-packages\sklearn\metr